# Planning Scenario Creation

This notebook creates reusable planning scenario bundles.

The workflow is:

1. read or prepare a site instance table;
2. define antenna, interface, device, metric, and connectivity profiles;
3. instantiate nodes with `PlanningScenario.add_node_instances`;
4. save TOML plus a companion node CSV with `PlanningScenario.save_bundle`.

`PlanningScenario` does not run geo feature extraction, metrics, or RPL. Those steps are done by `GraphPlanner` in the execution notebook.


In [ ]:
from pathlib import Path

import pandas as pd

from cisei_lib.planners import PlanningScenario

NOTEBOOK_DIR = Path.cwd()
POINTS_PATH = NOTEBOOK_DIR / "points.csv"
OUT_DIR = NOTEBOOK_DIR / "scenario_exports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WORKING_CRS = "EPSG:31982"
CONNECTED_SITES = {"torre", "itallia"}
RELAY_SITES = {"saturno", "deneka"}


## 1. Load Site Instances

The raw CSV is company/user input. The planning library consumes a standard table with `site_id`, position columns, optional installation height, and `device_profile`.


In [ ]:
points = pd.read_csv(POINTS_PATH)
points["site_id"] = points["id"].astype(str).str.strip()
points = points.drop(columns=["id"])
points["lat"] = points["lat"].astype(float)
points["lon"] = points["lon"].astype(float)
points["mount_height_m"] = points["ant_h"].astype(float)
points = points.drop(columns=["ant_h"])

points


## 2. Shared LTE Profiles

These helper functions keep the two examples consistent. A connected tower device has `connected=True` and `rank=0`. Leaf devices are unconnected and do not route.


In [ ]:
def add_common_lte_profiles(scenario: PlanningScenario) -> None:
    scenario.add_antenna_profile(
        "lte_tower_omni",
        {"kind": "omni", "model": "lte_tower_omni", "gain_dbi": 14.0},
    )
    scenario.add_antenna_profile(
        "lte_leaf_omni",
        {"kind": "omni", "model": "lte_leaf_omni", "gain_dbi": 6.0},
    )

    scenario.add_interface_profile(
        "lte_tower",
        {
            "tech": "lte",
            "freq_mhz": 400.0,
            "tx_power_dbm": 43.0,
            "antenna_id": "lte_tower_omni",
            "can_relay": False,
        },
    )
    scenario.add_interface_profile(
        "lte_leaf",
        {
            "tech": "lte",
            "freq_mhz": 400.0,
            "tx_power_dbm": 20.0,
            "antenna_id": "lte_leaf_omni",
            "can_relay": False,
        },
    )

    scenario.add_device_profile(
        "connected_lte_tower",
        {
            "kind": "tower",
            "connected": True,
            "can_route": True,
            "rank": 0.0,
            "interfaces": ["lte_tower"],
        },
    )
    scenario.add_device_profile(
        "lte_leaf_device",
        {
            "kind": "field",
            "connected": False,
            "can_route": False,
            "interfaces": ["lte_leaf"],
        },
    )

    scenario.add_metric_spec("lte", {"spec": "lte_tower_node"})


## 3. Scenario A: Single-Interface LTE

Every site has exactly one LTE interface. `torre` and `itallia` are connected sites. All other sites are LTE leaves.


In [ ]:
single = PlanningScenario(working_crs=WORKING_CRS)
add_common_lte_profiles(single)
single.add_connectivity_rule(
    {
        "kind": "rf",
        "tech": "lte",
        "source": "connected",
        "destination": "unconnected",
    }
)

single_instances = points.copy()
single_instances["device_profile"] = single_instances["site_id"].apply(
    lambda site_id: (
        "connected_lte_tower"
        if site_id in CONNECTED_SITES
        else "lte_leaf_device"
    )
)

single.add_node_instances(single_instances)
print("validation errors:", single.validate())
single.instance_table()


In [ ]:
single_toml, single_csv = single.save_bundle(
    OUT_DIR / "graph_planner_single_interface.toml"
)

print("TOML:", single_toml)
print("instances CSV:", single_csv)


## 4. Scenario B: LTE + Node-To-Node Relays

Most sites remain pure LTE leaves. Only `saturno` and `deneka` receive a second n2n interface and can route between LTE and n2n. This keeps the candidate graph controlled while testing multi-interface planning.


In [ ]:
two_if = PlanningScenario(working_crs=WORKING_CRS)
add_common_lte_profiles(two_if)

two_if.add_antenna_profile(
    "n2n_omni",
    {"kind": "omni", "model": "n2n_omni", "gain_dbi": 3.0},
)
two_if.add_interface_profile(
    "n2n_leaf",
    {
        "tech": "n2n",
        "freq_mhz": 915.0,
        "tx_power_dbm": 20.0,
        "antenna_id": "n2n_omni",
        "can_relay": False,
    },
)
two_if.add_interface_profile(
    "n2n_relay",
    {
        "tech": "n2n",
        "freq_mhz": 915.0,
        "tx_power_dbm": 23.0,
        "antenna_id": "n2n_omni",
        "can_relay": True,
    },
)
two_if.add_device_profile(
    "dual_lte_n2n_relay",
    {
        "kind": "field",
        "connected": False,
        "can_route": True,
        "interfaces": ["lte_leaf", "n2n_relay"],
    },
)
two_if.add_metric_spec("n2n", {"spec": "default"})
two_if.add_connectivity_rule(
    {
        "kind": "rf",
        "tech": "lte",
        "source": "connected",
        "destination": "unconnected",
    }
)
two_if.add_connectivity_rule(
    {
        "kind": "rf",
        "tech": "n2n",
        "source": "relay",
        "destination": "any",
        "degree": 3,
        "limit": 1000.0,
    }
)
two_if.add_connectivity_rule(
    {
        "kind": "internal",
        "enabled_when_device_can_route": True,
        "metric": 0.0,
    }
)


In [ ]:
two_if_instances = points.copy()

def two_if_profile(site_id: str) -> str:
    if site_id in CONNECTED_SITES:
        return "connected_lte_tower"
    if site_id in RELAY_SITES:
        return "dual_lte_n2n_relay"
    return "lte_leaf_device"

two_if_instances["device_profile"] = two_if_instances["site_id"].apply(two_if_profile)

two_if.add_node_instances(two_if_instances)
print("validation errors:", two_if.validate())
two_if.instance_table()


In [ ]:
two_if_toml, two_if_csv = two_if.save_bundle(
    OUT_DIR / "graph_planner_two_interface.toml"
)

print("TOML:", two_if_toml)
print("instances CSV:", two_if_csv)


## 5. Reload Check

This confirms the saved TOML files can load their companion instance CSV files automatically.


In [ ]:
loaded_single = PlanningScenario.from_toml(single_toml)
loaded_two_if = PlanningScenario.from_toml(two_if_toml)

print("single nodes:", len(loaded_single.site_nodes), "errors:", loaded_single.validate())
print("two-interface nodes:", len(loaded_two_if.site_nodes), "errors:", loaded_two_if.validate())
